# Grain-size statistics at real depth scale – events only

Plots your **pre-calculated** grain-size statistics (D10, D50, D90, Mean, Sorting)
for the two cores, **one figure per statistic per core** (10 figures), so each
can be placed next to the core photo.

* **Real scale:** with `SCALE = 1.0`, 1 cm of core = exactly 1 cm on the page.
  The PDF is saved at that exact size (no cropping). Place it at 100 %, line up
  the top edge of the data box with the top depth of the window on your photo
  (`DEPTH_WINDOW`), and every depth below lines up by itself.
* **Events only:** a sample is plotted when its middle depth lies inside an
  event interval (`EVENTS`). With `STYLE = 'step'` each sample is drawn over its
  real depth interval (e.g. 15.5–16.0 cm), cut at the event boundary.
* **No conversion:** every value is plotted in the unit of your file. Mean in φ
  is drawn with coarse to the right (`PHI_COARSE_RIGHT`).
* Cell 2 prints which column it used for each statistic, its unit, and the first
  rows. **Compare them with your csv before using the figures.**

Run the cells from top to bottom. All settings are at the top of Cell 1.

In [ ]:
# =============================================================================
# CELL 1 - SETTINGS + UPLOAD
# Grain-size statistics (already calculated) plotted at REAL depth scale,
# event intervals only, ONE figure per variable per core, ready to be placed
# next to a core photo.
# =============================================================================
import pandas as pd, numpy as np, re, os, glob, zipfile
import matplotlib, matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, MaxNLocator
from google.colab import files

# ---- 1. your files (words that appear in the file names) --------------------
FILES = {'GUAC-29A': 'stats 29A',
         'GUAC-24A': 'stats 24A'}

# ---- 2. event depths (top_cm, base_cm) - exactly as logged ------------------
EVENTS = {
 'GUAC-29A': {1:(13.40,14.10), 2:(15.10,19.10), 3:(22.60,30.15), 4:(31.30,33.20),
              5:(33.55,36.80), 6:(36.80,41.00), 7:(41.00,49.10), 8:(53.25,55.80),
              9:(56.80,58.10),10:(60.20,60.60),11:(61.20,64.10),12:(64.65,66.60),
             13:(67.15,67.50),14:(67.90,68.30),15:(70.80,71.15),16:(71.25,71.50),
             17:(73.15,75.40),18:(76.65,80.00),19:(82.45,87.75)},
 'GUAC-24A': {1:(11.75,11.90), 2:(12.90,17.40), 3:(19.35,26.20), 4:(27.50,27.85),
              5:(28.35,30.80), 6:(30.90,33.00), 7:(33.00,48.00), 8:(51.70,53.00),
             10:(55.70,55.85),11:(56.50,62.00),14:(63.50,63.75),15:(66.25,66.50),
             16:(66.70,67.10),17:(68.80,69.75),18:(70.15,70.55),19:(71.30,72.70),
             20:(75.20,79.80)},
}

# ---- 3. depth window of each figure = the part of the core in your photo ---
#         The plot's top edge is TOP cm and its bottom edge is BOTTOM cm.
DEPTH_WINDOW = {'GUAC-29A': (13.0, 88.0),
                'GUAC-24A': (11.0, 80.0)}

# ---- 4. real scale -----------------------------------------------------------
SCALE = 1.0          # cm on the page per cm of core. 1.0 = true size (1:1).
                     # 0.5 = half size. Use the SAME value as your photo.
PANEL_W_CM = 4.0     # width of the data box, cm
MARGIN_CM = dict(left=1.6, right=1.1, top=2.0, bottom=0.6)   # space for labels

# ---- 5. what to plot ----------------------------------------------------------
# Column names are found automatically. If one is wrong, write the exact name
# from your csv here, e.g. 'Mean': 'MEAN (φ)'.
COLUMNS = {'D10': None, 'D50': None, 'D90': None, 'Mean': None, 'Sorting': None}
# Units are read from the column name ('µm', 'phi', 'φ'). Nothing is converted.
# If a name has no unit, set it here: 'phi' or 'µm'.
UNITS   = {'D10': None, 'D50': None, 'D90': None, 'Mean': None, 'Sorting': 'phi'}
PHI_COARSE_RIGHT = True   # size in phi: coarse to the right (axis reversed)
STYLE = 'step'            # 'step'  = each sample drawn over its real interval
                          #           (e.g. a 15.5-16.0 cm sample spans 0.5 cm)
                          # 'point' = a dot at the middle of each sample
SHADE_EVENTS = True       # pale band behind each event interval
LABEL_EVENTS = True       # 'E03' next to each event

PALETTE = ['#1f77b4','#d62728','#2ca02c','#ff7f0e','#9467bd','#8c564b',
           '#e377c2','#17becf','#bcbd22','#7f3f00','#3f51b5','#00897b']

matplotlib.rcParams['pdf.fonttype'] = 42      # editable text in the PDF
matplotlib.rcParams['font.size'] = 8

print("Upload the grain-size statistics files of both cores.")
try: files.upload()
except Exception: pass
print("In /content:", [os.path.basename(f) for f in glob.glob('/content/*.csv') +
                       glob.glob('/content/*.xlsx')])

In [ ]:
# =============================================================================
# CELL 2 - READ THE FILES: depths, the five statistics, their units
# =============================================================================
def norm(x):
    return re.sub(r'[^a-z0-9]', '', str(x).lower())

def find_file(spec):
    """File whose name contains every word of spec (case, _ and - ignored)."""
    words = [norm(w) for w in re.split(r'[\s_\-]+', spec) if norm(w)]
    hits = [f for f in glob.glob('/content/*')
            if f.lower().endswith(('.csv', '.xlsx', '.xls'))
            and not os.path.basename(f).startswith(('GS_STATS_', 'EVENT_TABLE_'))
            and all(w in norm(os.path.basename(f)) for w in words)]
    # (GS_STATS_ / EVENT_TABLE_ files are written by the big notebook - skipped)
    assert hits, f"no uploaded file name contains all of {words}"
    return max(hits, key=os.path.getmtime)               # newest upload

def read_table(path):
    return (pd.read_excel(path) if path.lower().endswith(('.xlsx', '.xls'))
            else pd.read_csv(path, encoding='utf-8-sig'))

INTERVAL = re.compile(r'(\d+\.?\d*)\s*(?:-|–|to)\s*(\d+\.?\d*)')

def sample_depth(label):
    """'GUAC_29A 15.5-16.0cm' -> (15.5, 16.0)     'GUAC-24A-1G-1-W 13.0cm' -> (13.0, 13.0)"""
    s = str(label)
    m = [(float(a), float(b)) for a, b in INTERVAL.findall(s) if float(a) < float(b) <= 500]
    if m: return m[-1]
    m = re.findall(r'(\d+\.?\d*)\s*cm', s, re.I) or re.findall(r'\d+\.\d+', s)
    return (float(m[-1]), float(m[-1])) if m else (np.nan, np.nan)

def get_depths(df):
    """Top and bottom of every sample, from depth columns or from the labels."""
    cols = {norm(c): c for c in df.columns}
    top = next((cols[k] for k in cols if k.startswith(('depthtop', 'top'))), None)
    bot = next((cols[k] for k in cols if k.startswith(('depthbot', 'depthbase', 'bottom', 'base'))), None)
    if top and bot:
        return pd.to_numeric(df[top], errors='coerce'), pd.to_numeric(df[bot], errors='coerce')
    dcol = next((cols[k] for k in cols if 'depth' in k), None)
    if dcol is not None and pd.to_numeric(df[dcol], errors='coerce').notna().mean() > 0.9:
        v = pd.to_numeric(df[dcol], errors='coerce'); return v, v
    for c in df.columns:                                  # sample labels
        d = [sample_depth(v) for v in df[c].head(20)]
        if np.mean([np.isfinite(a) for a, _ in d]) > 0.5:
            d = np.array([sample_depth(v) for v in df[c]])
            print(f"   depths read from the labels in column '{c}'")
            return pd.Series(d[:, 0]), pd.Series(d[:, 1])
    raise ValueError(f"no depth found in columns {list(df.columns)}")

PATTERNS = {'D10': ('d10', 'dx10', 'dv10'), 'D50': ('d50', 'dx50', 'dv50', 'median'),
            'D90': ('d90', 'dx90', 'dv90'), 'Mean': ('mean', 'mz'),
            'Sorting': ('sorting', 'sd', 'sigma')}

def unit_of(colname):
    n = str(colname).lower()
    if 'phi' in n or 'φ' in n or '(f)' in n: return 'phi'
    if any(u in n for u in ('µm', 'μm', 'um', 'micron')): return 'µm'
    return None

def pick_column(df, var):
    """The column for one statistic; a phi-labelled one wins if there are two
    (Folk & Ward statistics are defined in phi)."""
    if COLUMNS.get(var): return COLUMNS[var]
    hits = [c for c in df.columns if norm(c).startswith(PATTERNS[var])]
    phi = [c for c in hits if unit_of(c) == 'phi']
    return (phi or hits or [None])[0]

DATA, UNIT_OF = {}, {}
for core, spec in FILES.items():
    path = find_file(spec)
    print(f"\n{core}  <-  {os.path.basename(path)}")
    raw = read_table(path)
    top, bot = get_depths(raw)
    out = pd.DataFrame({'top_cm': top.values, 'bot_cm': bot.values})
    for var in PATTERNS:
        c = pick_column(raw, var)
        if c is None:
            print(f"   {var:8s} not found - skipped"); continue
        out[var] = pd.to_numeric(raw[c], errors='coerce').values
        u = UNITS.get(var) or unit_of(c)
        if u is None:                       # name has no unit
            if var == 'Mean' and 'D50' in out:
                # read both ways; the one closer to D50 (µm) is right
                m, d50 = out[var].median(), out['D50'].median()
                u = 'µm' if abs(np.log(m / d50)) < abs(np.log(1000 * 2**-m / d50)) else 'phi'
            else:
                u = 'phi' if var == 'Sorting' else 'µm'
        UNIT_OF[(core, var)] = u
        print(f"   {var:8s} <- column '{c}'   unit: {u}   "
              f"range {out[var].min():.3g}-{out[var].max():.3g}")
    out['mid_cm'] = (out.top_cm + out.bot_cm) / 2
    out = out.dropna(subset=['mid_cm'])
    # repeated measurements of one sample -> averaged
    out = out.groupby(['top_cm', 'bot_cm'], as_index=False).mean().sort_values('mid_cm')
    DATA[core] = out.reset_index(drop=True)
    print(f"   {len(out)} samples, {out.top_cm.min():.2f}-{out.bot_cm.max():.2f} cm")
    print(f"   first rows - compare with your file:")
    print(out.head(3).round(3).to_string(index=False))

# one colour per event NUMBER, the same in both cores
ALL_IDS = sorted({e for c in EVENTS for e in EVENTS[c]})
EV_COLOR = {e: PALETTE[i % len(PALETTE)] for i, e in enumerate(ALL_IDS)}

In [ ]:
# =============================================================================
# CELL 3 - PLOT: one figure per statistic per core, at real depth scale
#
# Real scale: the data box is exactly (bottom - top) x SCALE cm tall, and the
# page is the data box + the margins. The figure is saved WITHOUT 'tight'
# cropping, so the PDF keeps that exact size. In Illustrator / PowerPoint:
# place the PDF at 100 %, then line up the data box's top edge with the TOP
# depth on your photo - every depth below lines up by itself.
# =============================================================================
CM = 1 / 2.54                                      # inches per cm
OUT = '/content/realscale'
os.makedirs(OUT, exist_ok=True)

def label_for(var, unit):
    name = {'D10': 'D10', 'D50': 'D50', 'D90': 'D90',
            'Mean': 'Mean grain size', 'Sorting': 'Sorting'}[var]
    return f"{name} ({'φ' if unit == 'phi' else unit})"

def event_samples(df, var, top, base):
    """Samples whose MIDDLE lies inside the event, with a value for var."""
    m = (df.mid_cm >= top) & (df.mid_cm <= base) & df[var].notna()
    return df[m]

def realscale_figure(core, var):
    df, unit = DATA[core], UNIT_OF[(core, var)]
    d_top, d_bot = DEPTH_WINDOW[core]
    box_h = (d_bot - d_top) * SCALE                # cm
    W = MARGIN_CM['left'] + PANEL_W_CM + MARGIN_CM['right']
    H = MARGIN_CM['top'] + box_h + MARGIN_CM['bottom']
    fig = plt.figure(figsize=(W * CM, H * CM))
    # data box placed in exact cm (fractions of the page)
    ax = fig.add_axes([MARGIN_CM['left'] / W, MARGIN_CM['bottom'] / H,
                       PANEL_W_CM / W, box_h / H])

    vals = []
    for e, (top, base) in sorted(EVENTS[core].items()):
        if base < d_top or top > d_bot: continue
        col = EV_COLOR[e]
        if SHADE_EVENTS:
            ax.axhspan(top, base, color=col, alpha=0.10, lw=0, zorder=0)
        seg = event_samples(df, var, top, base)
        if seg.empty: continue
        vals += list(seg[var])
        if STYLE == 'step' and (seg.bot_cm > seg.top_cm).all():
            # each sample = a vertical bar over its real interval, cut at the
            # event boundary; neighbours joined by a horizontal step
            ys, xs = [], []
            for _, r in seg.iterrows():
                y0, y1 = max(r.top_cm, top), min(r.bot_cm, base)
                ys += [y0, y1]; xs += [r[var], r[var]]
            ax.plot(xs, ys, color=col, lw=1.2, solid_joinstyle='miter')
        else:
            ax.plot(seg[var], seg.mid_cm, color=col, lw=0.9, marker='o',
                    ms=2.5, markeredgewidth=0)
        if LABEL_EVENTS:
            ax.text(1.02, (top + base) / 2, f'E{e:02d}', color=col, fontsize=6.5,
                    fontweight='bold', va='center', ha='left',
                    transform=ax.get_yaxis_transform(), clip_on=False)

    # depth axis: exact window, depth down, tick every 1 cm, label every 5 cm
    ax.set_ylim(d_bot, d_top)
    ax.yaxis.set_major_locator(MultipleLocator(5))
    ax.yaxis.set_minor_locator(MultipleLocator(1))
    ax.set_ylabel('Depth (cm)')
    # value axis: fitted to the event values
    if vals:
        lo, hi = np.nanmin(vals), np.nanmax(vals); pad = 0.06 * (hi - lo or 1)
        lo, hi = lo - pad, hi + pad
        reverse = unit == 'phi' and var != 'Sorting' and PHI_COARSE_RIGHT
        ax.set_xlim((hi, lo) if reverse else (lo, hi))
    ax.xaxis.set_major_locator(MaxNLocator(4))
    ax.xaxis.tick_top(); ax.xaxis.set_label_position('top')
    ax.set_xlabel(label_for(var, unit) +
                  ('\ncoarser →' if unit == 'phi' and var != 'Sorting' and PHI_COARSE_RIGHT else ''))
    # core name in the top-left corner of the page (0.25 cm from the edges)
    fig.text(0.25 / W, 1 - 0.25 / H, core, fontsize=8, fontweight='bold',
             ha='left', va='top')
    for s in ('right', 'bottom'): ax.spines[s].set_visible(False)

    name = f"{OUT}/{core}_{var}_realscale_{SCALE:g}x"
    fig.savefig(name + '.pdf', facecolor='white')            # NO bbox='tight'
    fig.savefig(name + '.png', dpi=200, facecolor='white')
    plt.close(fig)
    return name, box_h, W, H

for core in DATA:
    print(f"\n{core}: depth {DEPTH_WINDOW[core][0]}-{DEPTH_WINDOW[core][1]} cm "
          f"at {SCALE:g}:1")
    for var in ['D10', 'D50', 'D90', 'Mean', 'Sorting']:
        if var not in DATA[core]: continue
        name, box_h, W, H = realscale_figure(core, var)
        print(f"   {os.path.basename(name)}.pdf   page {W:.1f} x {H:.1f} cm, "
              f"data box {PANEL_W_CM:g} x {box_h:.1f} cm, "
              f"box top {MARGIN_CM['top']:g} cm below the page top")

# a quick look at one core in the notebook (small, NOT to scale)
core = list(DATA)[0]
fig, axs = plt.subplots(1, 5, figsize=(12, 6), sharey=True)
for ax, var in zip(axs, ['D10', 'D50', 'D90', 'Mean', 'Sorting']):
    if var not in DATA[core]: ax.axis('off'); continue
    for e, (t, b) in EVENTS[core].items():
        s = event_samples(DATA[core], var, t, b)
        ax.plot(s[var], s.mid_cm, '-o', ms=2, lw=0.8, color=EV_COLOR[e])
    ax.set_title(label_for(var, UNIT_OF[(core, var)]), fontsize=8)
    if UNIT_OF[(core, var)] == 'phi' and var != 'Sorting' and PHI_COARSE_RIGHT:
        ax.invert_xaxis()
axs[0].set_ylim(*DEPTH_WINDOW[core][::-1]); axs[0].set_ylabel('Depth (cm)')
fig.suptitle(f'{core} - preview only (the PDFs are the real-scale versions)')
plt.tight_layout(); plt.show()

In [ ]:
# =============================================================================
# CELL 4 - DOWNLOAD all real-scale figures (PDF + PNG) in one zip
# =============================================================================
ZIP = '/content/GrainSize_realscale.zip'
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob(f'{OUT}/*')):
        z.write(f, os.path.basename(f))
print(f"{ZIP}: {len(glob.glob(OUT + '/*'))} files")
files.download(ZIP)